# Classificazione con PySpark MLlib
### Dataset: `in-vehicle-coupon-recommendation.csv`

> **Obiettivo:** Costruire un modello predittivo di Machine Learning in Spark in grado di prevedere la probabilità che un guidatore accetti **(1)** o rifiuti **(0)** un coupon di sconto offerto durante la guida in base a diverse variabili, tra cui condizioni del viaggio, attributi del coupon e dati demografici e personali del guidatore.

### **0. Setup dell'ambiente**
La variabili d'ambiente vanno impostate ed eseguite prima che parta la JVM per prevenire errori che altrimenti compaiono più avanti.

In [1]:
import os
import sys
import shutil
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Su Windows: decommenta e adatta se winutils non e' gia' nelle variabili di sistema
# os.environ["HADOOP_HOME"] = r"C:\hadoop"
# os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

print("Interprete    :", sys.executable)
print("HADOOP_HOME   :", os.environ.get("HADOOP_HOME", "NON IMPOSTATO"))
print("SPARK_LOCAL_IP:", os.environ.get("SPARK_LOCAL_IP"))

Interprete    : C:\Users\Allysa\Documents\MachineLearning-in-Spark\.venv\Scripts\python.exe
HADOOP_HOME   : C:\hadoop
SPARK_LOCAL_IP: 127.0.0.1


### **1. Avvio di sessione Spark**
Inizializzazione della sessione Spark per il calcolo distribuito, configurando la memoria e le partizioni ottimali.

Esecuzione di un **smoke test** per verificareche il canale JVM driver <-> Python worker funzioni.

In [13]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("InVehicleCouponRecommendation")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.python.authenticate.socketTimeout", "60")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark  :", spark.version)
print("Shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark  : 4.2.0
Shuffle: 4


In [14]:
# Smoke test del canale JVM <-> Python worker
assert spark.sparkContext.parallelize([1, 2, 3]).map(lambda x: x * 2).collect() == [2, 4, 6]
print("Test Spark: Perfetto!")

Test Spark: Perfetto!


### **2. Caricamento del dataset**
Lettura del dataset e importazione all'interno di un DataFrame
`inferSchema=True`-> Spark leggerà il file due volte: una per analizzare il contenuto delle colonne e capire di che tipo sono, e secondo per il caricamento effettivo.

In [12]:
DATA_PATH = Path("data/in-vehicle-coupon-recommendation.csv")
if not DATA_PATH.exists():
     raise FileNotFoundError(f"File non esiste nel percorso specificato!")

In [5]:
df = spark.read.csv(str(DATA_PATH), header=True, inferSchema=True)

### **3. Analisi Esplorativa**
Analisi dello schema e verifica delle statistiche descrittive.
> **NB:** Il nome convenzionale è `import python.sql.functions as F` per assicurare che vengano usate le funzioni Spark e non quelle native Python.

### **4. Feature Engineering**
Comprende la **pulizia** (gestione dei valori mancanti e sbagliati e rimozione dei duplicati), **la conversione** (trasformazione delle variabili categoriche e assemblaggio di un vettore unico con `VectorAssembler` e preparazione del target.

> **NB:** La colonna `age` viene trattata come categorica perché nel dataset reale compare anche il valore `50plus`, quindi non è una variabile numerica pura.

### **5. Addestramento del modello**
Per avere un termine di paragone base (baseline), si addestra un primo modello di classificazione lineare tramite la `LinearRegression`.

 Successivamente si addestra un altro modello più avanzato con `RandomForest` pe4r catturare le relazioni complesse e non lineari.

Infine, si fa un confronto tra i due modelli per stabilire quale algoritmo sia più adatto al problema.

### **6. Tuning con Cross validation**
Per trovare i parametri perfetti senza rischiare l'overfitting, si testano diverse combinazioni di iperparametri tramite validazione incrociata.

### **7. Analisi delle Feature Importance**
Interrogare il modello finale per estrarre il peso decisionale di ogni feature, costruendo un grafico a barre per mostrare le più influenti.